# Imports

In [12]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

# Data

In [10]:
df = pd.read_csv('no_sales_df.csv')
df

,Unnamed: 0.1,Unnamed: 0,Order Date,Ship Date,Ship Mode,Segment,City,State,Country,Market,...,Discount,Profit,Shipping Cost,Order Priority,Price,Avg_Sales_Category,Avg_Sales_Country,Avg_Sales_Market,Avg_Sales_Region,Discount_value
0,0,234,2012-03-30,2012-04-01,First Class,Consumer,Cairo,Al Qahirah,Egypt,Africa,...,0.0,140.1600,399.96,Critical,637.350,467.858939,172.770678,170.868370,170.868370,0.0000
1,1,238,2014-12-23,2014-12-26,First Class,Consumer,Detroit,Michigan,United States,US,...,0.0,412.5394,397.52,High,226.670,416.248905,229.858001,229.858001,253.872674,0.0000
2,2,239,2014-07-04,2014-07-04,Same Day,Home Office,Seattle,Washington,United States,US,...,0.2,209.5800,396.92,High,399.200,467.858939,229.858001,229.858001,226.493233,479.0400
3,3,240,2014-11-21,2014-11-23,First Class,Consumer,New York City,New York,United States,US,...,0.0,327.5922,394.57,Critical,419.990,467.858939,229.858001,229.858001,238.336110,0.0000
4,4,241,2013-01-23,2013-01-27,Standard Class,Consumer,Baku,Baki,Azerbaijan,EMEA,...,0.0,946.6800,393.62,High,514.500,416.248905,194.190000,160.302508,160.302508,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50805,50805,51285,2014-06-19,2014-06-19,Same Day,Corporate,Kure,Hiroshima,Japan,APAC,...,0.0,4.5000,0.01,Medium,13.020,121.097120,403.150068,325.917481,362.835663,0.0000
50806,50806,51286,2014-06-20,2014-06-24,Standard Class,Consumer,Houston,Texas,United States,US,...,0.8,-1.1100,0.01,Medium,0.444,121.097120,229.858001,229.858001,253.872674,0.3552
50807,50807,51287,2013-12-02,2013-12-02,Same Day,Home Office,Oxnard,California,United States,US,...,0.0,11.2308,0.01,High,7.640,121.097120,229.858001,229.858001,226.493233,0.0000
50808,50808,51288,2012-02-18,2012-02-22,Standard Class,Home Office,Valinhos,São Paulo,Brazil,LATAM,...,0.0,2.4000,0.00,Medium,6.720,121.097120,225.832657,210.278334,240.919043,0.0000


In [34]:
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
numeric_features = df.select_dtypes(include=['number']).columns.tolist()
ohe_features = ['Segment', 'State', 'Country', 'Market', 'Region', 'Category', 'Sub-Category']
label_features = ['Ship Mode', 'Order Priority']

numeric_features = [col for col in numeric_features if col != 'Price']

encoder = OrdinalEncoder()
df[categorical_features] = encoder.fit_transform(df[categorical_features])

X = df.drop('Price', axis=1)
y = df['Price']
X

,Unnamed: 0.1,Unnamed: 0,Order Date,Ship Date,Ship Mode,Segment,City,State,Country,Market,...,Quantity,Discount,Profit,Shipping Cost,Order Priority,Avg_Sales_Category,Avg_Sales_Country,Avg_Sales_Market,Avg_Sales_Region,Discount_value
0,0,234,434.0,452.0,0.0,0.0,553.0,30.0,37.0,1.0,...,2,0.0,140.1600,399.96,0.0,467.858939,172.770678,170.868370,170.868370,0.0000
1,1,238,1421.0,1451.0,0.0,0.0,903.0,646.0,139.0,6.0,...,7,0.0,412.5394,397.52,1.0,416.248905,229.858001,229.858001,253.872674,0.0000
2,2,239,1252.0,1276.0,1.0,2.0,2932.0,1049.0,139.0,6.0,...,6,0.2,209.5800,396.92,1.0,467.858939,229.858001,229.858001,226.493233,479.0400
3,3,240,1389.0,1418.0,0.0,0.0,2288.0,703.0,139.0,6.0,...,3,0.0,327.5922,394.57,0.0,467.858939,229.858001,229.858001,238.336110,0.0000
4,4,241,731.0,753.0,3.0,0.0,258.0,110.0,8.0,3.0,...,4,0.0,946.6800,393.62,1.0,416.248905,194.190000,160.302508,160.302508,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50805,50805,51285,1237.0,1261.0,1.0,1.0,1724.0,410.0,65.0,0.0,...,5,0.0,4.5000,0.01,3.0,121.097120,403.150068,325.917481,362.835663,0.0000
50806,50806,51286,1238.0,1266.0,3.0,0.0,1399.0,982.0,139.0,6.0,...,1,0.8,-1.1100,0.01,3.0,121.097120,229.858001,229.858001,253.872674,0.3552
50807,50807,51287,1039.0,1062.0,1.0,2.0,2406.0,192.0,139.0,6.0,...,3,0.0,11.2308,0.01,1.0,121.097120,229.858001,229.858001,226.493233,0.0000
50808,50808,51288,397.0,413.0,3.0,2.0,3342.0,954.0,17.0,5.0,...,2,0.0,2.4000,0.00,3.0,121.097120,225.832657,210.278334,240.919043,0.0000


In [35]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)
X_temp

,Unnamed: 0.1,Unnamed: 0,Order Date,Ship Date,Ship Mode,Segment,City,State,Country,Market,...,Quantity,Discount,Profit,Shipping Cost,Order Priority,Avg_Sales_Category,Avg_Sales_Country,Avg_Sales_Market,Avg_Sales_Region,Discount_value
14233,14233,14702,657.0,679.0,3.0,0.0,2866.0,866.0,38.0,5.0,...,5,0.000,35.50000,20.25,1.0,121.097120,241.243075,210.278334,253.872674,0.000000
15789,15789,16261,1071.0,1096.0,2.0,1.0,3142.0,702.0,6.0,0.0,...,4,0.100,34.23600,17.50,3.0,416.248905,326.131778,325.917481,315.510356,30.855600
44647,44647,45126,116.0,120.0,0.0,2.0,151.0,1059.0,139.0,6.0,...,7,0.000,10.34880,1.21,1.0,121.097120,229.858001,229.858001,253.872674,0.000000
50514,50514,50994,411.0,429.0,3.0,1.0,2496.0,776.0,139.0,6.0,...,3,0.700,-1.78920,0.15,3.0,121.097120,229.858001,229.858001,238.336110,1.789200
32187,32187,32665,1324.0,1354.0,3.0,1.0,1399.0,982.0,139.0,6.0,...,8,0.200,11.55360,4.40,1.0,121.097120,229.858001,229.858001,253.872674,6.374400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11284,11284,11748,834.0,859.0,0.0,0.0,2431.0,768.0,74.0,3.0,...,2,0.700,-219.80400,27.29,0.0,121.097120,134.486640,160.302508,160.302508,87.091200
44732,44732,45211,598.0,620.0,3.0,0.0,1638.0,507.0,59.0,3.0,...,1,0.000,3.54000,1.20,3.0,121.097120,187.390626,160.302508,160.302508,0.000000
38158,38158,38636,858.0,884.0,3.0,1.0,3611.0,422.0,26.0,0.0,...,9,0.000,9.18000,2.57,3.0,121.097120,372.639375,325.917481,362.835663,0.000000
860,860,1234,1313.0,1342.0,2.0,1.0,1788.0,553.0,30.0,5.0,...,8,0.002,83.88576,180.01,0.0,467.858939,219.412894,210.278334,191.882166,1.551132


# Model

In [36]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

model = RandomForestRegressor(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_

/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/ilabarymov/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check i

In [37]:
best_model = RandomForestRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)

RandomForestRegressor(max_depth=30, max_features='sqrt', n_estimators=300,
                      n_jobs=-1, random_state=42)

In [38]:
y_test_pred = best_model.predict(X_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2_test = r2_score(y_test, y_test_pred)

print(f'\nТестовые метрики:')
print(f'RMSE (test): {rmse_test:.2f}')
print(f'R² (test): {r2_test:.4f}')


Тестовые метрики:
RMSE (test): 35.56
R² (test): 0.8674


In [39]:
feature_importances = best_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

print("\nВажность признаков:")
print(importance_df.head(10))

import joblib
joblib.dump(best_model, 'random_forest_price_prediction.pkl')


Важность признаков:
               Feature  Importance
1           Unnamed: 0    0.162208
0         Unnamed: 0.1    0.158617
16       Shipping Cost    0.137580
15              Profit    0.114225
13            Quantity    0.113554
18  Avg_Sales_Category    0.049124
22      Discount_value    0.038598
12        Sub-Category    0.038383
17      Order Priority    0.023310
11            Category    0.018972


['random_forest_price_prediction.pkl']